In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
import re

# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load dataset
df = pd.read_csv('../data/JobsDataset.csv')

# Check basic info
print("Dataset Shape:", df.shape)
print("Categories:", df['Query'].nunique())

# Clean text function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower() # Lowercase
    text = re.sub(r'<[^>]+>', ' ', text) # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # Remove special characters and numbers
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    return " ".join(tokens)

# Apply cleaning
df['cleaned_description'] = df['Description'].apply(clean_text)
print("Text cleaning complete!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\valen\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset Shape: (10000, 4)
Categories: 25
Text cleaning complete!


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Features and Target
X = df['cleaned_description']
y = df['Query']

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Vectorization using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Classifier (LinearSVC candidate model)
model = LinearSVC()
model.fit(X_train_tfidf, y_train)

# Evaluation
y_pred = model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Save Vectorizer and Model for App deployment
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
joblib.dump(model, '../models/skillmax_model.pkl')
print("Model and Vectorizer saved successfully in models/ folder!")

Accuracy: 0.6460

Classification Report:
                                precision    recall  f1-score   support

      Artificial Intelligence       0.76      0.80      0.78        76
            Big Data Engineer       0.52      0.45      0.48        64
             Business Analyst       0.66      0.77      0.71        79
Business Intelligence Analyst       0.75      0.70      0.73        74
              Cloud Architect       0.67      0.59      0.63       108
     Cloud Services Developer       0.62      0.53      0.57        79
                 Data Analyst       0.54      0.61      0.57        79
               Data Architect       0.61      0.42      0.50        80
                Data Engineer       0.49      0.42      0.45        77
         Data Quality Manager       0.65      0.65      0.65        79
               Data Scientist       0.67      0.72      0.69        80
    Data Visualization Expert       0.49      0.43      0.46        75
             Data Warehousing     